# Matrix Compression with SVD

**Idea:** a picture is just a matrix of numbers. The Singular Value Decomposition (SVD)
splits that matrix into *layers* ordered from most to least important.
Keeping only the first **k** layers gives a smaller matrix that still looks like the picture.

This notebook:
1. shows the idea on a tiny 3×3 matrix,
2. compresses three photos (shark, deer, leopard),
3. saves every figure into the `image/` folder, ready to drag into Canva.

Run it from top to bottom (**Run All**).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

IMAGE_DIR = Path("images")  # input photos
OUT_DIR = Path("image")     # figures for the slides
OUT_DIR.mkdir(exist_ok=True)

ANIMALS = {"Shark": "shark.jpg", "Deer": "deer.jpg", "Leopard": "leopard.jpg"}
COLORS = {"Shark": "#2a78d6", "Deer": "#eb6834", "Leopard": "#1baf7a"}
LINESTYLES = {"Shark": "-", "Deer": "--", "Leopard": ":"}

plt.rcParams.update({
    "font.size": 13, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e3e3e0", "axes.edgecolor": "#52514e",
    "figure.facecolor": "white",
})


def save(fig, name):
    """Save a figure into image/ (for Canva) and show it here."""
    fig.savefig(OUT_DIR / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 1. A tiny example: why a matrix can be compressed

Every row of this matrix is a multiple of `[1, 2, 3]`:

$$
A = \begin{bmatrix} 2 & 4 & 6 \\ 1 & 2 & 3 \\ 3 & 6 & 9 \end{bmatrix}
  = \begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix} \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}
$$

So 9 numbers can be stored as 6. SVD finds this automatically:
only the first singular value is non-zero, so **one layer** rebuilds the whole matrix.

In [ ]:
A = np.array([[2, 4, 6],
              [1, 2, 3],
              [3, 6, 9]], dtype=float)

U, S, Vt = np.linalg.svd(A)
print("singular values:", S.round(3))  # only the first one is non-zero

layer_1 = S[0] * np.outer(U[:, 0], Vt[0])  # σ1 · u1 · v1^T
print("rebuilt from 1 layer:\n", layer_1.round(3))

## 2. Load the photos and compute the SVD

A colour photo is 3 matrices (Red, Green, Blue). We compute the SVD of each one:

$$A = U \Sigma V^T = \sigma_1 u_1 v_1^T + \sigma_2 u_2 v_2^T + \cdots$$

The singular values $\sigma_1 \ge \sigma_2 \ge \dots$ say how important each layer is.

In [ ]:
def load(filename):
    """Load a photo as an (m, n, 3) matrix and compute the SVD of each colour."""
    img = np.asarray(Image.open(IMAGE_DIR / filename).convert("RGB"), dtype=float)
    svds = [np.linalg.svd(img[:, :, c], full_matrices=False) for c in range(3)]
    return img, svds


data = {name: load(filename) for name, filename in ANIMALS.items()}
for name, (img, _) in data.items():
    print(f"{name:8} {img.shape[0]} x {img.shape[1]} pixels, 3 colours")

## 3. Compress: keep only the first k layers

$$A_k = U_k \Sigma_k V_k^T = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$

- **Storage:** each layer needs $u_i$ ($m$ numbers), $v_i$ ($n$ numbers) and $\sigma_i$ (1 number),
  so $k$ layers need $k(m+n+1)$ numbers instead of $mn$.
- **Quality:** PSNR in dB (higher = closer to the original; above 30 dB looks very good).

In [ ]:
def compress(svds, k):
    """Keep the first k layers of each colour: A_k = U_k S_k V_k^T."""
    channels = [U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :] for U, S, Vt in svds]
    return np.clip(np.stack(channels, axis=2), 0, 255).astype(np.uint8)


def storage(img, k):
    """Fraction of the original storage: k(m + n + 1) / (m n)."""
    m, n, _ = img.shape
    return k * (m + n + 1) / (m * n)


def psnr(img, approx):
    """Quality in dB: 10 log10(255^2 / MSE)."""
    return 10 * np.log10(255 ** 2 / np.mean((img - approx) ** 2))


def break_even(img):
    """Largest k that still uses less storage than the original."""
    m, n, _ = img.shape
    return m * n // (m + n + 1)


for name, (img, _) in data.items():
    print(f"{name:8} compression only saves space up to k = {break_even(img)}")

### Check the theory (Eckart–Young)

The theorem says $A_k$ is the **best** rank-$k$ approximation, and its error is exactly
the size of the layers we threw away:
$\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \cdots}$

In [ ]:
img, svds = data["Shark"]
A = img[:, :, 0]          # red channel
U, S, Vt = svds[0]

print(f"{'k':>4} {'||A - A_k||':>14} {'sqrt(sum of thrown-away σ²)':>30}")
for k in [10, 50, 200]:
    A_k = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    print(f"{k:>4} {np.linalg.norm(A - A_k):>14.2f} {np.sqrt(np.sum(S[k:] ** 2)):>30.2f}")

## 4. Figures for the slides

Every cell below saves a PNG into `image/`.

### Slide idea: "SVD splits a picture into layers"
Top row: single layers. Bottom row: adding the layers up — the shark appears.

In [ ]:
gray = data["Shark"][0].mean(axis=2)  # grayscale version so the layers are easy to see
U, S, Vt = np.linalg.svd(gray, full_matrices=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col, i in enumerate([1, 2, 3, 10]):
    layer = S[i - 1] * np.outer(U[:, i - 1], Vt[i - 1])
    axes[0, col].imshow(layer, cmap="gray")
    axes[0, col].set_title(f"layer {i} only")
    total = U[:, :i] @ np.diag(S[:i]) @ Vt[:i, :]
    axes[1, col].imshow(np.clip(total, 0, 255), cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title(f"layers 1 to {i} added")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("SVD splits a picture into layers, most important first", fontsize=18)
fig.tight_layout()
save(fig, "01_svd_layers.png")

### Slide idea: "More layers = sharper picture, bigger file"
One figure per animal: the original next to k = 5, 20, 50, 100, 200.

In [ ]:
K_GRID = [5, 20, 50, 100, 200]

for name, (img, svds) in data.items():
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.6), gridspec_kw={"hspace": 0.35})
    axes[0, 0].imshow(img.astype(np.uint8))
    axes[0, 0].set_title("Original\n100% storage", fontsize=15)
    for ax, k in zip(axes.flat[1:], K_GRID):
        approx = compress(svds, k)
        ax.imshow(approx)
        ax.set_title(f"k = {k}\n{storage(img, k):.1%} storage · {psnr(img, approx):.1f} dB",
                     fontsize=15)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle(f"{name}: keeping k layers", fontsize=19)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.01, wspace=0.04)
    save(fig, f"02_grid_{name.lower()}.png")

### Slide idea: "Small details come last"
Zoom into the shark's teeth and the leopard's face.

In [ ]:
ZOOMS = {  # (top, bottom, left, right) as fractions of the picture
    "Shark": (0.38, 0.70, 0.40, 0.60),
    "Leopard": (0.26, 0.62, 0.38, 0.58),
}

for name, (y0, y1, x0, x1) in ZOOMS.items():
    img, svds = data[name]
    m, n, _ = img.shape
    rows, cols = slice(int(y0 * m), int(y1 * m)), slice(int(x0 * n), int(x1 * n))
    panels = [(f"k = {k}", compress(svds, k)) for k in [20, 50, 200]]
    panels.append(("Original", img.astype(np.uint8)))

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    for ax, (title, picture) in zip(axes, panels):
        ax.imshow(picture[rows, cols])
        ax.set_title(title, fontsize=15)
        ax.axis("off")
    fig.suptitle(f"{name} close-up: fine details appear last", fontsize=18)
    fig.tight_layout()
    save(fig, f"03_zoom_{name.lower()}.png")

### Slide idea: "Why it works: singular values drop fast"
Each curve is divided by its first singular value so the three photos can be compared.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))
for name, (img, svds) in data.items():
    S = np.mean([s for _, s, _ in svds], axis=0)  # average of R, G, B
    ax.semilogy(np.arange(1, len(S) + 1), S / S[0], color=COLORS[name],
                ls=LINESTYLES[name], lw=2.5, label=name)
ax.set(xlim=(1, 400), ylim=(1e-3, 1.3), xlabel="layer i", ylabel="σᵢ / σ₁  (log scale)",
       title="Importance of each layer drops quickly")
ax.legend(frameon=False)
fig.tight_layout()
save(fig, "04_singular_values.png")

### Slide idea: "How many layers are worth it?"
Quality against storage for the three photos. Higher and further left is better.

In [ ]:
K_CURVE = list(range(1, 20)) + list(range(20, 401, 10))

fig, ax = plt.subplots(figsize=(12, 6.5))
for name, (img, svds) in data.items():
    x = [storage(img, k) * 100 for k in K_CURVE]
    y = [psnr(img, compress(svds, k)) for k in K_CURVE]
    ax.plot(x, y, color=COLORS[name], ls=LINESTYLES[name], lw=2.5, label=name)
ax.axhline(30, color="#52514e", ls="--", lw=1)
ax.text(1, 30.4, "30 dB ≈ looks very good", color="#52514e")
ax.set(xlim=(0, 100), xlabel="storage used (% of original)",
       ylabel="quality, PSNR (dB)", title="Quality vs. storage")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save(fig, "05_quality_vs_storage.png")

### Slide idea: results table

In [ ]:
K_TABLE = [5, 20, 50, 100, 200]

rows = []
for name, (img, svds) in data.items():
    rows.append([name] + [f"{psnr(img, compress(svds, k)):.1f} dB" for k in K_TABLE])
img = data["Shark"][0]
rows.append(["Storage (shark)"] + [f"{storage(img, k):.1%}" for k in K_TABLE])

for row in rows:
    print("  ".join(f"{cell:>15}" for cell in row))

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis("off")
table = ax.table(cellText=rows, colLabels=["Image"] + [f"k = {k}" for k in K_TABLE],
                 loc="center", cellLoc="center")
table.scale(1, 2.2)
table.set_fontsize(14)
for (r, _), cell in table.get_celld().items():
    cell.set_edgecolor("#d8d8d4")
    if r == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f0efec")
ax.set_title("Quality (PSNR) for each k", fontsize=16)
save(fig, "06_results_table.png")

## 5. Summary

- A picture is a matrix; SVD writes it as a sum of layers $\sigma_i u_i v_i^T$, most important first.
- Keeping $k$ layers stores $k(m+n+1)$ numbers instead of $mn$.
- By the Eckart–Young theorem, the first $k$ layers are the **best possible** rank-$k$ approximation.
- Smooth pictures (deer: sky and blurred background) compress better than busy ones
  (shark bubbles, leopard spots).
- Limitation: SVD keeps what has the most *energy*, not what people find important — small details
  like teeth come last.

For a live demo, run `python demo.py`.